# Phase 6 — validation against IRS SOI 2023

This notebook tests whether the trained CPS model tracks independently published
IRS aggregates by income band.

It keeps the two sources separate:

- **CPS model rate:** survey-weighted mean model prediction for CPS filer units.
- **CPS observed rate:** survey-weighted mean observed CPS effective rate.
- **IRS-published rate:** IRS all-return net income tax divided by IRS all-return
  adjusted gross income for the same published band.

The CPS series use **whole-return pre-tax income** for band placement. The IRS
series use **adjusted gross income**. Those are related but different income
concepts, so the chart is a credibility check rather than an identity test.

Sources:

- IPUMS CPS ASEC 2024, income year 2023; frozen tax-unit tables in this repository.
- IRS Publication 1304, Tax Year 2023, Table 1.1:
  `https://www.irs.gov/pub/irs-soi/23in11si.xls`
- IRS Publication 1304, Tax Year 2023, Table 3.3:
  `https://www.irs.gov/pub/irs-soi/23in33ar.xls`

The known-bad processed IRS master file is never read. Negative rates are never
clipped.

In [ ]:
from __future__ import annotations

import math
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import Markdown, display
from matplotlib.ticker import FuncFormatter
from sklearn.metrics import mean_absolute_error, r2_score

ROOT = Path.cwd().resolve()
if not (ROOT / "model_interface.py").is_file():
    ROOT = ROOT.parent
assert (ROOT / "model_interface.py").is_file(), "Run this notebook from the repository root."
sys.path.insert(0, str(ROOT))

import model_interface as mi

TABLE11_PATH = ROOT / "data" / "raw" / "irs_soi_table11_2023.xls"
TABLE33_PATH = ROOT / "data" / "raw" / "irs_soi_table33_2023.xls"
FIGURE_PATH = ROOT / "reports" / "figures" / "phase6_irs_validation.png"
TABLE_PATH = ROOT / "reports" / "phase6_comparison.csv"
SUMMARY_PATH = ROOT / "reports" / "phase6_summary.md"

for required in (TABLE11_PATH, TABLE33_PATH, mi.TRAIN_PATH, mi.TEST_PATH):
    assert required.is_file(), f"Required input is missing: {required}"

pd.set_option("display.max_columns", 20)
pd.set_option("display.width", 180)
pd.set_option("display.float_format", lambda value: f"{value:,.3f}")

## Primary validation: the held-out CPS test set

The model is loaded only through `model_interface`. The notebook does not load
the artifact directly and does not reconstruct the encoder. The primary
validation remains the untouched 20% test split; full-population results below
are descriptive because the training rows are in-sample.

In [ ]:
manifest = mi._manifest()
mi._validate_frozen_csv(mi.TRAIN_PATH, manifest["sha256"]["train.csv"])
mi._validate_frozen_csv(mi.TEST_PATH, manifest["sha256"]["test.csv"])

train = pd.read_csv(mi.TRAIN_PATH)
test = pd.read_csv(mi.TEST_PATH)
expected_columns = list(mi.FEATURE_COLS) + [manifest["target"], manifest["weight_retained"]]
assert list(train.columns) == expected_columns
assert list(test.columns) == expected_columns

model = mi._load_model()
X_test = test.loc[:, list(mi.FEATURE_COLS)]
y_test = test[manifest["target"]].to_numpy(dtype=float)
w_test = test[manifest["weight_retained"]].to_numpy(dtype=float)
test_predictions = np.asarray(model.predict(X_test), dtype=float)
assert np.isfinite(test_predictions).all()

def weighted_mean(values: np.ndarray, weights: np.ndarray) -> float:
    return float(np.average(np.asarray(values, dtype=float), weights=np.asarray(weights, dtype=float)))

def weighted_r2(observed: np.ndarray, predicted: np.ndarray, weights: np.ndarray) -> float:
    center = weighted_mean(observed, weights)
    residual = np.sum(weights * (observed - predicted) ** 2)
    total = np.sum(weights * (observed - center) ** 2)
    return float(1 - residual / total)

held_out_metrics = pd.DataFrame(
    {
        "Measure": ["R²", "MAE (rate points)", "Survey-weighted R²", "Survey-weighted MAE (rate points)"],
        "Held-out test result": [
            r2_score(y_test, test_predictions),
            mean_absolute_error(y_test, test_predictions),
            weighted_r2(y_test, test_predictions, w_test),
            weighted_mean(np.abs(y_test - test_predictions), w_test),
        ],
    }
)

logged = mi._metrics()["metrics"]
assert math.isclose(held_out_metrics.loc[0, "Held-out test result"], logged["test"]["R2"], abs_tol=1e-3)
assert math.isclose(held_out_metrics.loc[1, "Held-out test result"], logged["test"]["MAE"], abs_tol=1e-3)
display(held_out_metrics.style.format({"Held-out test result": "{:.4f}"}))
print(f"Held-out rows: {len(test):,}")
print(f"Prediction range: {test_predictions.min():.2f}% to {test_predictions.max():.2f}%")

## IRS-published benchmark from the raw workbooks

Table 1.1 supplies the all-return AGI denominator. Table 3.3 supplies “total
income tax minus refundable credits,” the all-return net-tax numerator. Both
workbooks state that money amounts are in thousands of dollars, so their scale
cancels in the rate calculation.

Only the ordinary, non-accumulated income-band rows are used. The notebook
asserts the sheet names, headers, exact band sequence, and all-return counts
before joining them.

In [ ]:
IRS_BANDS = [
    "No adjusted gross income",
    "$1 under $5,000",
    "$5,000 under $10,000",
    "$10,000 under $15,000",
    "$15,000 under $20,000",
    "$20,000 under $25,000",
    "$25,000 under $30,000",
    "$30,000 under $40,000",
    "$40,000 under $50,000",
    "$50,000 under $75,000",
    "$75,000 under $100,000",
    "$100,000 under $200,000",
    "$200,000 under $500,000",
    "$500,000 under $1,000,000",
    "$1,000,000 under $1,500,000",
    "$1,500,000 under $2,000,000",
    "$2,000,000 under $5,000,000",
    "$5,000,000 under $10,000,000",
    "$10,000,000 or more",
]

table11 = pd.read_excel(TABLE11_PATH, sheet_name="TBL11", header=None, engine="xlrd")
table33 = pd.read_excel(TABLE33_PATH, sheet_name="TBL33", header=None, engine="xlrd")

assert "Table 1.1." in str(table11.iat[0, 0])
assert "Tax Year 2023" in str(table11.iat[0, 0])
assert "All returns" in str(table11.iat[2, 1])
assert "Adjusted gross" in str(table11.iat[3, 3])
assert str(table11.iat[5, 3]).strip() == "Amount"

assert "Table 3.3." in str(table33.iat[0, 0])
assert "Tax Year 2023" in str(table33.iat[0, 0])
assert "Total income tax" in str(table33.iat[2, 122])
assert "refundable credits" in str(table33.iat[2, 122])
assert str(table33.iat[6, 123]).strip() == "Amount"

def ordinary_band_rows(raw: pd.DataFrame) -> pd.DataFrame:
    starts = raw.index[raw.iloc[:, 0].astype(str).eq(IRS_BANDS[0])].tolist()
    assert starts, "The first ordinary income band was not found."
    result = raw.iloc[starts[0] : starts[0] + len(IRS_BANDS)].copy()
    assert result.iloc[:, 0].astype(str).tolist() == IRS_BANDS
    return result

agi_rows = ordinary_band_rows(table11)
tax_rows = ordinary_band_rows(table33)

irs_agi = pd.DataFrame(
    {
        "Income band": agi_rows.iloc[:, 0].astype(str).to_numpy(),
        "IRS all returns": pd.to_numeric(agi_rows.iloc[:, 1]).to_numpy(dtype=float),
        "IRS AGI ($ thousands)": pd.to_numeric(agi_rows.iloc[:, 3]).to_numpy(dtype=float),
    }
)
irs_tax = pd.DataFrame(
    {
        "Income band": tax_rows.iloc[:, 0].astype(str).to_numpy(),
        "IRS all returns (Table 3.3)": pd.to_numeric(tax_rows.iloc[:, 1]).to_numpy(dtype=float),
        "IRS net tax ($ thousands)": pd.to_numeric(tax_rows.iloc[:, 123]).to_numpy(dtype=float),
    }
)

irs = irs_agi.merge(irs_tax, on="Income band", how="inner", validate="one_to_one")
assert irs["Income band"].tolist() == IRS_BANDS
irs["IRS return-count difference"] = (
    irs["IRS all returns (Table 3.3)"] - irs["IRS all returns"]
)
# The two published sample estimates differ by one return in the $2m–$5m
# band. Preserve that audit fact instead of silently forcing the counts equal.
assert irs["IRS return-count difference"].abs().max() <= 1
count_differences = irs.loc[
    irs["IRS return-count difference"].ne(0),
    [
        "Income band",
        "IRS all returns",
        "IRS all returns (Table 3.3)",
        "IRS return-count difference",
    ],
]
assert (irs["IRS AGI ($ thousands)"] != 0).all()
irs["IRS-published rate (%)"] = (
    100 * irs["IRS net tax ($ thousands)"] / irs["IRS AGI ($ thousands)"]
)
assert np.isfinite(irs["IRS-published rate (%)"]).all()

display(
    Markdown(
        "**Published count audit:** the band labels align exactly. "
        "The two IRS tables differ by one estimated return in one band:"
    )
)
display(count_differences)

display(
    irs[
        [
            "Income band",
            "IRS all returns",
            "IRS AGI ($ thousands)",
            "IRS net tax ($ thousands)",
            "IRS-published rate (%)",
        ]
    ].style.format(
        {
            "IRS all returns": "{:,.0f}",
            "IRS AGI ($ thousands)": "{:,.0f}",
            "IRS net tax ($ thousands)": "{:,.0f}",
            "IRS-published rate (%)": "{:.2f}",
        }
    )
)

### Band alignment

The 19 ordinary bands in Tables 1.1 and 3.3 align exactly—no IRS band is
dropped, split, or combined. Their sample estimates differ by one return in the
`$2,000,000 under $5,000,000` band (203,229 in Table 1.1 and 203,230 in Table
3.3); the notebook preserves and displays that published discrepancy. It does
not alter either table's amount.

For CPS placement, zero whole-return income is shown beside the IRS “No
adjusted gross income” band; positive CPS income uses the same published dollar
cut points. The labels remain source-specific because CPS whole-return pre-tax
income is not IRS AGI.

## Secondary descriptive check: full weighted CPS filer population

Train and test are recombined only for this descriptive view. Every CPS rate is
weighted by `asecwt`; the survey weight never enters the model. Empty CPS
brackets remain missing rather than being invented as zero.

In [ ]:
frozen = pd.concat(
    [train.assign(_split="train"), test.assign(_split="test")],
    ignore_index=True,
)
assert len(frozen) == manifest["rows"]["full"]
assert (frozen["unit_inctot"] >= 0).all()

frozen["CPS model rate (%)"] = np.asarray(
    model.predict(frozen.loc[:, list(mi.FEATURE_COLS)]),
    dtype=float,
)
assert np.isfinite(frozen["CPS model rate (%)"]).all()

positive_edges = [
    0, 5_000, 10_000, 15_000, 20_000, 25_000, 30_000, 40_000, 50_000,
    75_000, 100_000, 200_000, 500_000, 1_000_000, 1_500_000, 2_000_000,
    5_000_000, 10_000_000, np.inf,
]
positive_labels = IRS_BANDS[1:]

frozen["Income band"] = pd.Series(pd.NA, index=frozen.index, dtype="object")
zero_income = frozen["unit_inctot"].eq(0)
frozen.loc[zero_income, "Income band"] = IRS_BANDS[0]
frozen.loc[~zero_income, "Income band"] = pd.cut(
    frozen.loc[~zero_income, "unit_inctot"],
    bins=positive_edges,
    labels=positive_labels,
    right=False,
).astype("object")
assert frozen["Income band"].notna().all()

def aggregate_cps_band(frame: pd.DataFrame, band: str) -> dict[str, float | int | str]:
    group = frame.loc[frame["Income band"].eq(band)]
    if group.empty:
        return {
            "Income band": band,
            "CPS sample rows": 0,
            "CPS weighted filer population": 0.0,
            "CPS model rate (%)": np.nan,
            "CPS observed rate (%)": np.nan,
        }
    weights = group["asecwt"].to_numpy(dtype=float)
    return {
        "Income band": band,
        "CPS sample rows": int(len(group)),
        "CPS weighted filer population": float(weights.sum()),
        "CPS model rate (%)": weighted_mean(group["CPS model rate (%)"].to_numpy(), weights),
        "CPS observed rate (%)": weighted_mean(group["eff_rate"].to_numpy(), weights),
    }

cps = pd.DataFrame([aggregate_cps_band(frozen, band) for band in IRS_BANDS])
comparison = cps.merge(
    irs[
        [
            "Income band",
            "IRS all returns",
            "IRS AGI ($ thousands)",
            "IRS net tax ($ thousands)",
            "IRS-published rate (%)",
        ]
    ],
    on="Income band",
    how="inner",
    validate="one_to_one",
)
assert comparison["Income band"].tolist() == IRS_BANDS
assert len(comparison) == len(IRS_BANDS)

TABLE_PATH.parent.mkdir(parents=True, exist_ok=True)
comparison.to_csv(TABLE_PATH, index=False)

comparison_display = comparison[
    [
        "Income band",
        "CPS model rate (%)",
        "CPS observed rate (%)",
        "IRS-published rate (%)",
        "CPS sample rows",
        "CPS weighted filer population",
        "IRS all returns",
    ]
]
display(
    comparison_display.style.format(
        {
            "CPS model rate (%)": "{:.2f}",
            "CPS observed rate (%)": "{:.2f}",
            "IRS-published rate (%)": "{:.2f}",
            "CPS sample rows": "{:,.0f}",
            "CPS weighted filer population": "{:,.0f}",
            "IRS all returns": "{:,.0f}",
        },
        na_rep="—",
    )
)

## Comparison chart

The three series are not averaged together. The zero line is explicit, and
negative values use the same visual treatment as positive values.

In [ ]:
SHORT_LABELS = [
    "No AGI", "$1–5k", "$5–10k", "$10–15k", "$15–20k", "$20–25k",
    "$25–30k", "$30–40k", "$40–50k", "$50–75k", "$75–100k",
    "$100–200k", "$200–500k", "$500k–1m", "$1–1.5m", "$1.5–2m",
    "$2–5m", "$5–10m", "$10m+",
]

x = np.arange(len(comparison))
fig, ax = plt.subplots(figsize=(15.5, 8.5))
fig.patch.set_facecolor("#FAFAF8")
ax.set_facecolor("#FAFAF8")

series = [
    ("CPS model rate — survey-weighted mean", "CPS model rate (%)", "#356AA0", "o", "-"),
    ("CPS observed rate — survey-weighted mean", "CPS observed rate (%)", "#C16A3A", "s", "-"),
    ("IRS-published rate — net tax ÷ AGI", "IRS-published rate (%)", "#222222", "D", "--"),
]
for label, column, color, marker, linestyle in series:
    ax.plot(
        x,
        comparison[column].to_numpy(dtype=float),
        label=label,
        color=color,
        marker=marker,
        linestyle=linestyle,
        linewidth=2.1,
        markersize=5.5,
        zorder=3,
    )

ax.axhline(0, color="#777777", linewidth=1.0, zorder=1)
ax.grid(axis="y", color="#D9D9D4", linewidth=0.8)
ax.grid(axis="x", visible=False)
ax.set_xticks(x, SHORT_LABELS, rotation=38, ha="right")
ax.yaxis.set_major_formatter(FuncFormatter(lambda value, _: f"{value:.0f}%"))
ax.set_ylabel("Effective federal tax rate")
ax.set_xlabel("Published dollar bands; CPS uses whole-return pre-tax income, IRS uses AGI")
ax.set_title(
    "The CPS model follows the published progression, with expected edge gaps",
    loc="left",
    fontsize=17,
    fontweight="semibold",
    pad=18,
)
ax.text(
    0,
    1.015,
    "Tax year 2023 · CPS rates are weighted by ASEC survey weights · Negative rates are retained",
    transform=ax.transAxes,
    ha="left",
    va="bottom",
    fontsize=10.5,
    color="#555555",
)
ax.legend(frameon=False, loc="upper left", ncol=1)
for spine in ("top", "right", "left"):
    ax.spines[spine].set_visible(False)
ax.spines["bottom"].set_color("#999999")
ax.tick_params(axis="y", length=0)

missing_cps = comparison["CPS sample rows"].eq(0)
if missing_cps.any():
    first_missing = int(np.flatnonzero(missing_cps.to_numpy())[0])
    ax.axvspan(first_missing - 0.45, len(comparison) - 0.55, color="#E9E9E4", alpha=0.6, zorder=0)
    ax.text(
        first_missing,
        ax.get_ylim()[0] + 0.8,
        "No CPS sample observations",
        fontsize=9,
        color="#666666",
        ha="left",
        va="bottom",
    )

fig.text(
    0.075,
    0.012,
    "Sources: IPUMS CPS ASEC 2024 (income year 2023); IRS SOI Publication 1304, Tables 1.1 and 3.3. "
    "IRS rate is aggregate net income tax after refundable credits divided by aggregate AGI.",
    ha="left",
    va="bottom",
    fontsize=8.8,
    color="#666666",
)
fig.subplots_adjust(left=0.075, right=0.985, top=0.84, bottom=0.23)
FIGURE_PATH.parent.mkdir(parents=True, exist_ok=True)
fig.savefig(FIGURE_PATH, dpi=220, facecolor=fig.get_facecolor())
plt.show()
print(f"Saved chart: {FIGURE_PATH.relative_to(ROOT)}")

## Plain-language findings

The summary below is generated from the comparison table so its cited values
cannot drift away from the chart.

In [ ]:
def band_result(name: str) -> pd.Series:
    return comparison.loc[comparison["Income band"].eq(name)].iloc[0]

def format_rate(value: float) -> str:
    return f"{value:.1f}%".replace("-", "−")

bottom = band_result("$1 under $5,000")
middle = band_result("$50,000 under $75,000")
top_with_cps = comparison.loc[comparison["CPS sample rows"].gt(0)].iloc[-1]
empty_top = comparison.loc[comparison["CPS sample rows"].eq(0), "Income band"].tolist()

full_weights = frozen["asecwt"].to_numpy(dtype=float)
full_model_mean = weighted_mean(frozen["CPS model rate (%)"].to_numpy(), full_weights)
full_observed_mean = weighted_mean(frozen["eff_rate"].to_numpy(), full_weights)

irs_total_agi = float(table11.loc[table11.iloc[:, 0].eq("All returns"), 3].iloc[0])
irs_total_net_tax = float(table33.loc[table33.iloc[:, 0].eq("All returns, total"), 123].iloc[0])
irs_overall_rate = 100 * irs_total_net_tax / irs_total_agi

summary = f"""# Phase 6 presentation summary

The held-out model remains accurate at the individual-filer level: R² is
{held_out_metrics.loc[0, "Held-out test result"]:.3f} and the mean absolute
error is {held_out_metrics.loc[1, "Held-out test result"]:.2f} percentage
points across {len(test):,} untouched CPS filer units.

As a secondary descriptive check, the survey-weighted CPS model and observed
rates trace the same broad progression as the IRS-published series. In the
$50,000–$75,000 band, the model predicts
{format_rate(middle["CPS model rate (%)"])}, the CPS observed rate is
{format_rate(middle["CPS observed rate (%)"])}, and the IRS-published net-tax
rate is {format_rate(middle["IRS-published rate (%)"])}.

Negative bottom-end rates are a finding, not an error. For the $1–$5,000 band,
the CPS model rate is {format_rate(bottom["CPS model rate (%)"])}, the CPS
observed rate is {format_rate(bottom["CPS observed rate (%)"])}, and the
IRS-published rate is {format_rate(bottom["IRS-published rate (%)"])}.
Refundable credits can exceed income tax owed, making net federal income tax
negative; no series is clipped.

The top end is where survey data are least representative. The highest band
with any CPS observation is {top_with_cps["Income band"]}, with only
{int(top_with_cps["CPS sample rows"]):,} sampled filer unit(s); there are no CPS
observations in {", ".join(empty_top)}. IRS administrative returns continue
through those brackets. This missing and sparse coverage—caused by survey
undersampling and top-coding of very high incomes—is the expected top-band
understatement, not evidence that the IRS series is wrong.

Across the full weighted CPS population, the model mean is
{format_rate(full_model_mean)} and the observed CPS mean is
{format_rate(full_observed_mean)}. The IRS all-return aggregate rate is
{format_rate(irs_overall_rate)}. These overall numbers are kept separate: CPS
values are weighted means of filer-level rates, while the IRS value is
aggregate net tax divided by aggregate AGI.

Method note: CPS brackets use whole-return pre-tax income; IRS brackets use
adjusted gross income. IRS includes all returns, while the frozen CPS table
excludes dependent filers. The comparison is therefore an external trend
check, not an assertion that the sources measure an identical population.
"""

SUMMARY_PATH.write_text(summary)
display(Markdown(summary))
print(f"Saved table: {TABLE_PATH.relative_to(ROOT)}")
print(f"Saved summary: {SUMMARY_PATH.relative_to(ROOT)}")

## Interpretation guardrails

- **No source blending:** every table column and chart series names either CPS
  or IRS explicitly. No CPS and IRS value is averaged together.
- **Survey weighting:** every full-population CPS band rate uses `asecwt`.
- **Negative rates:** retained throughout, without clipping or error styling.
- **Top-income coverage:** blank CPS points mean no sampled filer units, not a
  zero rate.
- **Different rate constructions:** CPS columns are weighted means of
  filer-level rates; IRS is the published aggregate net-tax-to-AGI ratio.
- **Different income concepts:** CPS bands use whole-return pre-tax income;
  IRS bands use adjusted gross income.